# Grafo combinado em lote

Esta pipeline processa todos os casos de `sample/cases.csv` ou uma seleção de IDs.
Gera `<case_id>-nodes.csv`, `<case_id>-edges.csv` e `_resumo.csv` em `data/processed/`.
O código reutilizável fica em `src/grafo_lote/`. O notebook original
`grafo_combinado.ipynb` permanece preservado para inspecionar um caso.

Execute as células em ordem usando um kernel com as dependências de
`src/grafo_lote/requirements.txt` e o recurso NLTK `punkt_tab` instalado.


In [ ]:
import logging
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(p for base in (Path.cwd(), *Path.cwd().parents)
            for p in (base, base / "project1")
            if (p / "src" / "grafo_lote").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from grafo_lote.lote import COLUNAS_RESUMO, executar_lote

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
print("Projeto:", ROOT)


## Configuração

Use `CASE_IDS = None` para todos os casos. Para um subconjunto, use, por exemplo,
`CASE_IDS = ["PMC5137649_01", "PMC3437073_01"]`.
Altere `SAIDA` para preservar resultados anteriores. Cada execução substitui o
resumo e os CSVs dos casos exportados; arquivos de outros casos permanecem no disco.


In [ ]:
CASOS = ROOT / "sample" / "cases.csv"
SAIDA = ROOT / "data" / "processed"
CASE_IDS = None


## Processamento e exportação

O lote carrega os gazetteers MeSH e anatômico, HMM e stopwords uma única vez.
Cada caso tem seu próprio grafo e normalizador. Falhas individuais são registradas
e o processamento continua. As verificações de evidência e domínio viram contadores
no relatório; elas não interrompem o caso com `assert`.


In [ ]:
registros = executar_lote(cases_csv=CASOS, output=SAIDA, case_ids=CASE_IDS)
resumo = pd.DataFrame(registros, columns=COLUNAS_RESUMO)
print(f"{len(resumo)} casos registrados. Relatório: {SAIDA / '_resumo.csv'}")
display(resumo)


## Conferência dos resultados

`ok`: exportação concluída sem inconsistências; `com_inconsistencias`: CSVs
exportados com problemas de evidência ou domínio; `erro`: falha individual.
Contadores vazios indicam que a etapa não foi concluída. Erros globais, como
entrada ou recursos ausentes, são apresentados pela célula de execução.


In [ ]:
display(resumo["status"].value_counts().rename_axis("status").to_frame("casos"))
contadores = ["nos", "arestas", "evidencias_desalinhadas", "arestas_fora_do_dominio"]
display(resumo[contadores].apply(pd.to_numeric, errors="coerce").sum().to_frame("total"))
pendencias = resumo.loc[resumo["status"] != "ok"]
if not pendencias.empty:
    display(pendencias)


## Exemplo de output

Prévia dos arquivos gerados para o primeiro caso exportado nesta execução.


In [ ]:
exportados = resumo.loc[resumo["status"].isin(["ok", "com_inconsistencias"]), "case_id"]
if not exportados.empty:
    exemplo = exportados.iloc[0]
    for sufixo in ("nodes", "edges"):
        caminho = SAIDA / f"{exemplo}-{sufixo}.csv"
        print(caminho)
        display(pd.read_csv(caminho).head(10))
else:
    print("Nenhum caso foi exportado nesta execução; consulte o relatório.")
